# MetaJudge: Selective Abstention (Family B)

**Task:** `metajudge_abstention` | **Version:** v4.1
**Axis:** Monitoring (Nelson & Narens 1990)
**Items:** 72 clean selective abstention items
**Score:** Anchor-normalized UWAA → float in [0, 1]

In [ ]:
# Cell 1 — Imports & Path Setup
import subprocess, sys, os, json
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'matplotlib', 'seaborn'])
from dataclasses import dataclass

# --- Kaggle input paths ---
# Package: metajudge scoring + grading engine
_pkg_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v2"  # v2 fallback,
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

# Data: benchmark items, registry, clean manifest
_data_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v3",
    "/kaggle/input/metajudge-data-v2"  # v2 fallback,
    "data",  # local development
]
DATA_ROOT = None
for _p in _data_paths:
    if os.path.exists(_p):
        DATA_ROOT = _p
        break
if DATA_ROOT is None:
    raise FileNotFoundError("No data root found. Add metajudge-data-v3 as Kaggle input.")

# Kaggle Benchmarks SDK
import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats, assertions

# Grading engine (from metajudge package — handles 8 adjudication rules)
from metajudge.scoring.grading_v2 import grade_item, load_registry

# Family B scoring — corrective-answer + acceptable-alternative handling
from metajudge.scoring.abstention_metrics import score_family_b_item_v2, compute_uwaa

print(f"Data root: {DATA_ROOT}")
print("MetaJudge benchmark ready.")

# Family C scoring + task helpers
from metajudge.tasks.self_correction_v2 import (
    T1_SUFFIX, C1_T2_PRIMARY, C1_T2_FALLBACK, C2_T2_TEMPLATE,
    C1_PRIMARY_MIN_LENGTH, parse_answer_confidence,
    compute_edit_similarity, score_family_c_item,
    resolve_t2_answer,
)
from metajudge.scoring.self_correction_v2 import classify_transition


OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Cell 2 — Scoring Formulas & Anchor Normalization
import numpy as np

ANCHOR_B_FLOOR = 0.60   # Empirical: random uniform actions → UWAA ≈ 0.60
ANCHOR_B_CEIL  = 1.00   # Perfect abstention

def normalize(score, floor, ceil):
    return max(0.0, min(1.0, (score - floor) / (ceil - floor)))

print(f"Scoring: score_family_b_item_v2 (package), anchor B=[{ANCHOR_B_FLOOR}, {ANCHOR_B_CEIL}]")

In [ ]:
# Cell 3 — Response Schemas

@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)


@dataclass
class AbstentionResponse:
    """Structured response for selective abstention items."""
    decision: str = "answer"       # answer | clarify | verify | abstain
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)

In [ ]:
# Cell 4 — Load Datasets & Registry
import pandas as pd

# Calibration items (Family A)
with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    all_cal_items = json.load(f)

# Family B selective abstention items
with open(os.path.join(DATA_ROOT, "family_b_pilot_v2.json")) as f:
    all_fb_items = json.load(f)

# Clean subset manifest — excludes suspect items
with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_items"])
fb_excluded = set(manifest["family_b"]["excluded_items"])

cal_items = [it for it in all_cal_items if it["item_id"] not in cal_excluded]
fb_items = [it for it in all_fb_items if it["item_id"] not in fb_excluded]

cal_df = pd.DataFrame(cal_items)
fb_df = pd.DataFrame(fb_items)

# Grading registry (8 adjudication rules for deterministic answer matching)
REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

print(f"Calibration: {len(cal_df)} clean items (excluded {len(cal_excluded)} from {len(all_cal_items)})")
print(f"Family B:    {len(fb_df)} clean items (excluded {len(fb_excluded)} from {len(all_fb_items)})")
print(f"Registry:    {len(REGISTRY)} grading rules loaded")

# Family C self-correction items (v3 dataset)
fc_path = os.path.join(DATA_ROOT, "family_c_items.json")
if os.path.exists(fc_path):
    with open(fc_path) as f:
        all_fc_items = json.load(f)
    fc_excluded = set(manifest.get("family_c", {}).get("excluded_items", []))
    fc_items = [it for it in all_fc_items if it["item_id"] not in fc_excluded]
    fc_df = pd.DataFrame(fc_items)
    fc_answer_key = {it["item_id"]: it for it in all_fc_items}
    print(f"Family C:    {len(fc_df)} clean items (excluded {len(fc_excluded)} from {len(all_fc_items)})")
else:
    fc_items, fc_df, fc_answer_key = [], pd.DataFrame(), {}
    print("Family C:    not found (v2 dataset -- skipping)")


In [ ]:
# Cell 6 — Task: Selective Abstention (Axis II — Control)

VALID_ACTIONS = {"answer", "clarify", "verify", "abstain"}

# Build lookup for item metadata needed by score_family_b_item_v2
_FB_META = {it["item_id"]: it for it in fb_items}

@kbench.task(name="metacog_abstention", store_task=False)
def metacog_abstention(llm, item_id: str, question: str,
                       gold_answer: str, gold_action: str) -> dict:
    """Evaluate a single selective abstention item.

    Uses score_family_b_item_v2 from the metajudge package — handles
    corrective answers on false-premise items and acceptable alternatives.
    Returns dict with item-level audit data.
    """
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            f"Question:\n{question}\n\n"
            "Choose exactly one action:\n"
            '- "answer": Provide your best answer\n'
            '- "clarify": Ask one specific clarifying question\n'
            '- "verify": Request external verification\n'
            '- "abstain": Decline to answer (genuinely unanswerable)\n\n'
            "Return valid structured output with keys: "
            "decision, answer, confidence, clarification_request, "
            "verification_target, abstention_reason"
        )
        response = llm.prompt(prompt, schema=AbstentionResponse)

    # Normalise decision
    decision = str(response.decision).strip().lower()
    if decision not in VALID_ACTIONS:
        decision = "abstain"

    # Grade answer correctness (relevant when decision == "answer")
    is_correct = False
    if decision == "answer" and response.answer:
        result = grade_item(item_id, response.answer, REGISTRY,
                            gold_answer=gold_answer)
        is_correct = result["correct"]

    # Score via full v2 scorer (corrective answers + acceptable alternatives)
    meta = _FB_META.get(item_id, {})
    utility = score_family_b_item_v2(
        model_decision=decision,
        model_answer=str(response.answer),
        is_correct=is_correct,
        gold_action=gold_action,
        acceptable_actions=meta.get("acceptable_actions", [gold_action]),
        is_false_presupposition=meta.get("is_false_presupposition", False),
    )

    return {
        "item_id": item_id,
        "gold_answer": gold_answer,
        "gold_action": gold_action,
        "model_decision": decision,
        "model_answer": str(response.answer),
        "confidence": round(max(0.0, min(1.0, float(response.confidence))), 4),
        "is_correct": is_correct,
        "utility": round(utility, 4),
    }

In [ ]:
# Cell 6 — Main Task: metajudge_abstention

@kbench.task(name="metajudge_abstention")
def metajudge_abstention(llm) -> float:
    """Family B — Selective Abstention (Monitoring).
    
    Returns anchor-normalized UWAA score.
    """
    fb_eval = fb_df[["item_id", "question", "gold_answer", "gold_action"]].copy()

    with kbench.client.enable_cache():
        fb_runs = metacog_abstention.evaluate(
            llm=[llm], evaluation_data=fb_eval,
            n_jobs=8, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(fb_eval),
            max_attempts=1,
        )

    records = [r.result for r in fb_runs if r.result is not None]
    audit = pd.DataFrame(records)

    uwaa = compute_uwaa(audit["utility"].tolist())
    normalized = normalize(uwaa, ANCHOR_B_FLOOR, ANCHOR_B_CEIL)

    act_acc = (audit["model_decision"] == audit["gold_action"]).mean()
    print(f"Abstention: UWAA={uwaa:.4f} Act Acc={act_acc:.1%} Normalized={normalized:.3f} [{len(audit)} items]")

    # Action distribution
    from collections import Counter
    actions = Counter(audit["model_decision"].tolist())
    print(f"  Actions: {dict(actions)}")

    # Export audit CSV
    audit.to_csv(os.path.join(OUTPUT_DIR, "family_b_item_audit.csv"), index=False)

    # Export full responses JSON
    import json as _json
    with open(os.path.join(OUTPUT_DIR, "family_b_full_responses.json"), "w") as f:
        _json.dump(records, f, indent=2, default=str)

    return normalized

In [ ]:
# Cell 7 — Run + Submit
metajudge_abstention.run(kbench.llm)
%choose metajudge_abstention

## Methodology

**Axis II — Selective Abstention (Monitoring).** Per-item utility from 5×4 payoff matrix mapping
(model action × gold action) → [−1, +1]. UWAA = (mean utility + 1) / 2. 72 clean items.
Handles corrective answers on false-presupposition items and acceptable alternative actions.
Normalized to [0, 1] using anchor floor 0.60 (empirical random uniform baseline) and ceiling 1.0.

Equal weighting across tasks is provably optimal at small n (Davis-Stober 2011; Dawes 1979).
Anchor normalization follows BIG-Bench (Srivastava et al. 2023).